# (Reference solutions) 07_time_series_diagnostics

> This is the **full reference-solution edition** of the matching main notebook. Try the exercises yourself first, then compare. All explanations and answers are original to this project.

# Week 7 — Time-Series Diagnostics

> Part of the open-source teaching project **quant-math-roadmap**.
> For **education and research methodology only** — not investment advice; no result here represents a profitable or investable strategy.

## Learning objectives

- Compare the statistical behavior of price series and return series.
- Compute and interpret the autocorrelation function (ACF).
- Compute rolling volatility and observe volatility clustering.
- Compare stationary (AR(1)) and non-stationary (random walk) series.

## Estimated study time

About 8–10 hours.

## Prerequisites

- Returns from Week 1
- Random variables from Week 3

## External resources

- [Forecasting: Principles and Practice, the Pythonic Way](https://otexts.com/fpppy/)
- [Penn State STAT 510 Applied Time Series Analysis](https://online.stat.psu.edu/stat510/)

> External resources are linked for reference only; this project does not reproduce any copyrighted course material.

In [ ]:
# Teaching style setup (deterministic look, consistent figures)
import matplotlib as _mpl
_mpl.rcParams['axes.unicode_minus'] = False
_mpl.rcParams['figure.figsize'] = (8.5, 4.5)
_mpl.rcParams['savefig.dpi'] = 100
import numpy as _np
_np.random.seed(0)  # belt-and-braces; library functions take explicit seeds

## Concepts

### Stationarity

A **stationary** series has statistical properties (mean, variance, autocorrelation) that do not change over time. Price series are usually **non-stationary** (they trend and drift); return series are usually **much closer to stationary**.

### The autocorrelation function (ACF)

The ACF measures how a series correlates with its own lagged values:

$$ \rho_k = \frac{\operatorname{Cov}(x_t, x_{t-k})}{\operatorname{Var}(x_t)}. $$

**White noise** has an ACF close to 0 at every nonzero lag.

### Why random splits do not apply

Time series are ordered. A random train/test split lets the model 'see the future', creating look-ahead bias — the central theme of Week 8.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from quant_math_roadmap.data import (
    SyntheticConfig, generate_correlated_prices,
    generate_ar1_series, generate_random_walk,
)
from quant_math_roadmap.finance.returns import simple_returns
from quant_math_roadmap.time_series.diagnostics import (
    adf_stationarity_test, autocorrelation_function,
    rolling_volatility,
)

config = SyntheticConfig(n_assets=1, n_periods=756, seed=21,
                         vol_regime_multiplier=2.0)
prices = generate_correlated_prices(config).iloc[:, 0]
returns = simple_returns(prices)

### Prices vs returns

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].plot(prices.index, prices.values)
axes[0].set_title('Price series (usually non-stationary: it trends)')
axes[0].set_ylabel('Price')
axes[1].plot(returns.index, returns.values)
axes[1].set_title('Return series (closer to stationary, but with volatility clustering)')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Daily return')
plt.tight_layout()
plt.show()

### The ADF stationarity test

In [ ]:
price_adf = adf_stationarity_test(prices)
return_adf = adf_stationarity_test(returns)
print('Price series  ADF p-value :', round(price_adf['p_value'], 4))
print('Return series ADF p-value :', round(return_adf['p_value'], 4))
print('A small p-value = evidence against a unit root (leaning stationary).')

### The autocorrelation function

In [ ]:
acf_returns = autocorrelation_function(returns, max_lag=20)
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.bar(acf_returns.index, acf_returns.values)
ax.set_title('Autocorrelation function (ACF) of returns')
ax.set_xlabel('Lag')
ax.set_ylabel('Autocorrelation')
plt.show()
print('The return ACF is mostly near 0 at nonzero lags — close to white noise.')

### Rolling volatility and volatility clustering

In [ ]:
roll_vol = rolling_volatility(returns, window=40)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(roll_vol.index, roll_vol.values, label='40-period rolling volatility')
ax.set_title('Rolling volatility: volatility clustering')
ax.set_xlabel('Date')
ax.set_ylabel('Rolling standard deviation')
ax.legend()
plt.show()
print('The synthetic data adds a volatility regime shift in the second half — clearly visible here.')

### Stationary vs non-stationary: AR(1) vs a random walk

In [ ]:
ar1 = generate_ar1_series(600, phi=0.6, seed=5)
walk = generate_random_walk(600, drift=0.0, seed=5)

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].plot(ar1.index, ar1.values)
axes[0].set_title('AR(1), phi=0.6 (stationary: reverts to its mean)')
axes[0].set_ylabel('Value')
axes[1].plot(walk.index, walk.values)
axes[1].set_title('Random walk (non-stationary: it drifts and does not come back)')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Value')
plt.tight_layout()
plt.show()
print('AR(1)       ADF p-value:', round(adf_stationarity_test(ar1)['p_value'], 4))
print('Random walk ADF p-value:', round(adf_stationarity_test(walk)['p_value'], 4))

## Exercises

Work through these in order. **Basic exercises** consolidate the definitions, **applied exercises** are hands-on coding, and the **reflection question** connects the mathematics to backtesting and research methodology.

> The main notebook ships runnable starter code for each coding exercise. Full reference answers live in the matching `_solution` notebook under `notebooks/en/solutions/`.

### Basic exercises

1. Define stationarity in your own words, and explain why prices are usually non-stationary.
2. What does the ACF of white noise look like?
3. Explain what volatility clustering is.

### Applied exercises

In [ ]:
# Applied exercise 1: generate three AR(1) series with phi=0.0, phi=0.5, and phi=0.9,
# compute each one's lag-1 autocorrelation, and confirm it is close to phi.
from quant_math_roadmap.time_series.diagnostics import autocorrelation
for phi in [0.0, 0.5, 0.9]:
    series = generate_ar1_series(4000, phi=phi, seed=1)
    ac1 = autocorrelation(series, 1)
    print(f'phi={phi}: lag-1 autocorr = {ac1:.3f}')

In [ ]:
# Applied exercise 2: compute the ACF for the price series and the return series, and compare them.
acf_price = autocorrelation_function(prices, max_lag=20)
print('Price lag-1 autocorrelation :', round(acf_price.iloc[1], 4))
print('Return lag-1 autocorrelation:', round(acf_returns.iloc[1], 4))
print('Prices are highly autocorrelated (non-stationary); returns are close to white noise.')

### Reflection question

1. Given that the return ACF is close to 0 almost everywhere, what does this imply for strategies that try to predict future returns from past returns?

## Quiz (self-check)
Answer the multiple-choice questions, then run the next cell to check yourself. Answers are stored as hashes, not plaintext.

**Q1. A stationary series is characterized by?**
- A. Prices always rise
- B. Statistical properties (mean, variance, autocorrelation) do not change over time
- C. No volatility at all
- D. No autocorrelation at all

**Q2. The ACF of white noise at nonzero lags should be?**
- A. Close to 0
- B. Close to 1
- C. Increasing with lag
- D. All negative

**Q3. AR(1): x_t = φx_{t−1} + ε_t is stationary under which condition?**
- A. φ > 0
- B. |φ| < 1
- C. φ = 1
- D. φ > 1

**Q4. 'Volatility clustering' refers to?**
- A. Returns concentrating near the mean
- B. High-volatility and low-volatility periods each arriving in clusters
- C. Prices clustering at round-number levels
- D. Zero autocorrelation

In [ ]:
my_answers = {1: 'B', 2: 'A', 3: 'B', 4: 'B'}

import hashlib as _hashlib
_expected = {1: '8652b328d7d88f86', 2: '917be34eb394105b', 3: 'cb788c82b06bc75e', 4: '65bc6a6a84b4ae5a'}
_n_correct = 0
for _q, _ans in my_answers.items():
    if _ans is None:
        print(f'Q{_q}: unanswered')
        continue
    _h = _hashlib.sha256(f'qmr-w7-q{_q}-{str(_ans).strip().upper()}'.encode()).hexdigest()[:16]
    _ok = _h == _expected[_q]
    _n_correct += int(_ok)
    print(f'Q{_q}: ' + ('✔ correct' if _ok else '✘ incorrect'))
print(f'Score: {_n_correct} / {len(my_answers)}')

### Explanations

- **Q1 → B**: Stationarity is time-invariance of the statistical properties; the series itself can still fluctuate randomly.
- **Q2 → A**: White noise is by definition uncorrelated across periods; the theoretical ACF is 0 everywhere except lag 0.
- **Q3 → B**: With |φ|<1 shocks decay away; φ=1 is a random walk (non-stationary).
- **Q4 → B**: Returns themselves are nearly uncorrelated, but their magnitude is highly autocorrelated — storms follow storms.

## Common mistakes

- **Modeling the non-stationary price series directly instead of converting to returns first.**
- **Using a random train/test split on time series.**
- **Backfilling the leading NaNs of a rolling window with future values.**
- **Over-interpreting weak return autocorrelation as a profitable signal.**

## After this week, you should be able to

- [ ] Explain stationarity and judge prices vs returns.
- [ ] Compute and interpret the ACF.
- [ ] Compute rolling volatility and recognize volatility clustering.
- [ ] Explain why random splits are unsuitable for time series.

## References and attribution

- Every explanation, example and exercise in this notebook is **original** to this project.
- Recommended external resources: [`docs/resources.md`](../../../docs/resources.md).
- Concept notes: [`docs/math/`](../../../docs/math/) and [`docs/finance/`](../../../docs/finance/).

### Privacy and disclaimer

- This notebook contains no real personal information.
- This notebook uses only reproducible synthetic data and needs no network access.
- This notebook makes no claim of real-world trading profitability.